In [ ]:
import pandas as pd

def calculate_precision(df, true_rating_col, predicted_rating_col, relevance_threshold):
    """
    Calculate precision as:
    Precision = Relevant Recommended Items / Total Recommended Items.

    Parameters:
        df (pd.DataFrame): The dataset containing true and predicted ratings.
        true_rating_col (str): Column name for true ratings.
        predicted_rating_col (str): Column name for predicted ratings.
        relevance_threshold (float): threshold to determine relevance.

    Returns:
        float: Precision value.
    """
    print("Relevance threshold: " + str(relevance_threshold))

    # Determine relevant items based on true ratings
    df['is_relevant'] = df[true_rating_col] >= relevance_threshold

    # Determine predicted relevant items based on predicted ratings
    df['predicted_relevant'] = df[predicted_rating_col] >= relevance_threshold

    # Calculate Relevant Recommended Items (both relevant & predicted as relevant)
    relevant_recommended = df[(df['is_relevant']) & (df['predicted_relevant'])].shape[0]

    # Calculate Total Recommended Items (all predicted as relevant)
    total_recommended = df['predicted_relevant'].sum()

    # Calculate Precision
    precision = (relevant_recommended / total_recommended if total_recommended > 0 else 0)

    return precision

############## EXAMPLE #############
# Calculate precision
relevance_threshold = ratings_matrix['rating'].quantile(0.90)
precision = calculate_precision(temp_po, 'rating', 'pred', relevance_threshold)
print(f"Precision: {precision:.4f}")


In [ ]:
def compute_recall_at_k(actual_interactions, recommendations, k=10):
    """
    Computes Recall@K for a set of users.

    Args:
        actual_interactions (dict): Mapping of user_id -> set of relevant item_ids (ground truth).
        recommendations (dict): Mapping of user_id -> list of recommended item_ids.
        k (int): Number of top recommendations to consider.

    Returns:
        float: Mean Recall@K across users.
    """
    recall_scores = []

    for user_id, actual_items in actual_interactions.items():
        if user_id in recommendations:
            recommended_items = recommendations[user_id][:k]
            if actual_items:
                hits = len(set(recommended_items) & set(actual_items))
                recall = hits / len(actual_items)
                recall_scores.append(recall)

    if recall_scores:
        mean_recall = sum(recall_scores) / len(recall_scores)
    else:
        mean_recall = 0.0

    print(f"Recall@{k}: {mean_recall:.4f}")
    return mean_recall


In [ ]:
import numpy as np
import pandas as pd

def compute_novelty(data: pd.DataFrame, final_recommendations: pd.DataFrame) -> tuple:
    """
    Computes the novelty and serendipity of the recommended articles using the pre-calculated popularity column.

    Parameters:
        data (pd.DataFrame): DataFrame containing all articles, including the popularity column.
        final_recommendations (pd.DataFrame): DataFrame containing recommended articles.

    Returns:
        tuple: (Average Novelty Score, Average Serendipity Score)
    """
    # Step 1: Aggregate and normalize popularity to ensure it stays between 0 and 1
    min_popularity = data['popularity'].min()
    max_popularity = data['popularity'].max()

    if max_popularity == min_popularity:
        data['normalized_popularity'] = 0.5  # If all popularity values are the same, assign a neutral value
    else:
        data['normalized_popularity'] = (data['popularity'] - min_popularity) / (max_popularity - min_popularity)
    
    # Step 2: Map normalized popularity values
    article_popularity = data.groupby('article_id')['normalized_popularity'].mean().to_dict()  # Ensure unique IDs
    
    # Step 3: Compute novelty scores (inverse of normalized popularity)
    novelty_scores = {k: 1 - v for k, v in article_popularity.items()}  # Less popular = Higher novelty

    # Step 4: Map novelty scores to recommended articles
    final_recommendations['novelty_score'] = final_recommendations['article_id'].map(novelty_scores).fillna(1)  # Unseen articles get max novelty

    # Step 5: Compute relevance score (Normalize ratings)
    if 'rating' in final_recommendations.columns:
        max_rating = final_recommendations['rating'].max()
        final_recommendations['relevance_score'] = final_recommendations['rating'] / max_rating if max_rating > 0 else 0
    else:
        final_recommendations['relevance_score'] = 1  # If no ratings, assume max relevance
    
    # Step 6: Compute serendipity as novelty * relevance
    avg_serendipity = (final_recommendations['novelty_score'] * final_recommendations['relevance_score']).mean()
    
    # Step 7: Compute average novelty
    avg_novelty = final_recommendations['novelty_score'].mean()
    
    return avg_novelty, avg_serendipity

# ------------------------------------------------------------------------------
# Example Usage:
# ------------------------------------------------------------------------------
novelty_score, serendipity_score = compute_novelty(interaction_data.copy(), final_recommendations.copy())

print(f"✅ Fixed Average Novelty Score: {novelty_score:.4f}")
print(f"✅ Fixed Average Serendipity Score: {serendipity_score:.4f}")


Diversity measures how different the recommended items are from each other. It’s typically calculated using pairwise similarity between items in the recommendations.

Steps:

Calculate pairwise similarity for the features column. Compute the average dissimilarity (1 - similarity) for all pairs of recommendations.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def calculate_diversity(recommended_items, item_features, item_id_col="article_id"):
    """
    Calculates diversity of recommendations based on item feature similarity.

    Args:
        recommended_items (DataFrame): DataFrame containing recommended items.
        item_features (list): List of column names representing item features.
        item_id_col (str): Column name in `final_rec` and `tada1` representing item IDs.

    Returns:
        float: Diversity score (higher means more diverse recommendations).
    """

    # Ensure we have enough items to calculate diversity
    if recommended_items.shape[0] < 2:
        return 1.0  # Maximum diversity if only one or zero items are recommended

    # Compute pairwise cosine similarity
    similarity_matrix = cosine_similarity(recommended_items[['Boring presentation',
         'Graphics',
         'Readability',
         'Consistency',
         'Images',
         'Bullets',
         'Text size',
         'Text heavy',
         'Tables',
         'Agenda',
         'Infographics',
         'Positioning',
         'Presentation length',
         'General tips',
         'Presentation skills',
         'Powerpoint knowledge',
         'tips and tricks',
         'Explanation of the problem']])

    # Get upper triangle values (excluding diagonal)
    sim_values = similarity_matrix[np.triu_indices(len(recommended_items), k=1)]

    # Calculate diversity (1 - mean similarity)
    diversity_score = 1 - np.mean(sim_values) if len(sim_values) > 0 else 1.0
    
    return round(diversity_score,2)


calculate_diversity(final_recommendations, item_features)

Types of Coverage:
User Coverage:

The percentage of users for whom the system can provide recommendations.
If coverage = 1, it means the system has made recommendations for 100% of the users.
Formula:

User Coverage
=
Users with Recommendations
Total Users
User Coverage=
Total Users
Users with Recommendations
​

Item Coverage:

The percentage of items that appear in at least one recommendation list.
If coverage = 1, it means every item in the catalog has been recommended at least once.
Formula:

Item Coverage
=
Items in Recommendations
Total Items
Item Coverage=
Total Items
Items in Recommendations
​

Example Interpretation:
User Coverage = 1:
The system is able to generate recommendations for all users in the dataset.
Item Coverage = 1:
The system has recommended all items from the catalog to at least one user.
Why Is Coverage Important?
Coverage is critical in recommendation systems for the following reasons:

Inclusivity: Ensures that all users or items are considered, preventing bias.
Diversity: Higher item coverage generally correlates with more diverse recommendations.

In [ ]:
def compute_item_coverage(interaction_data, final_recommendations):
    """
    Computes item coverage: the proportion of unique recommended items
    compared to all unique items in the dataset.

    Args:
        interaction_data (pd.DataFrame): DataFrame containing all interactions (must include 'article_id').
        final_recommendations (pd.DataFrame): DataFrame of recommendations (must include 'article_id').

    Returns:
        float: Item coverage value.
    """
    total_items = interaction_data['article_id'].nunique()
    items_in_recommendations = final_recommendations['article_id'].nunique()
    item_coverage = items_in_recommendations / total_items

    print(f"Total items in dataset: {total_items}")
    print(f"Items in recommendations: {items_in_recommendations}")
    print(f"Item Coverage: {item_coverage:.4f}")

    return item_coverage
